In [2]:
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
import matplotlib.pyplot as plt

2026-02-24 20:45:05.061047: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-24 20:45:05.140238: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-24 20:45:06.723660: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
data , info = tfds.load("voc/2007" , with_info=True)

2026-02-24 20:45:08.843346: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [4]:
def normal(example):
    img = tf.image.resize(example['image'], (125,125)) / 255.0
    bbox = example['objects']['bbox'][0]
    label = tf.cast(example['objects']['label'][0], tf.float32)
    y = tf.concat([bbox, [label]], axis=0)
    return img, y

In [5]:
train = data["train"]
train = train.map(normal).shuffle(500).batch(8)
test = data["validation"]
test = test.map(normal).batch(8)

In [6]:
model = keras.Sequential()
model.add(keras.layers.Input(shape=(125,125,3)))
model.add(keras.layers.Conv2D(16,3 , activation="relu", padding="same"))
model.add(keras.layers.MaxPool2D(2,2))
model.add(keras.layers.Conv2D(32,3 , activation="relu", padding="same"))
model.add(keras.layers.MaxPool2D(2,2))
model.add(keras.layers.Conv2D(64,3 , activation="relu", padding="same"))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(70 , activation="relu"))
model.add(keras.layers.Dense(5))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 125, 125, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 62, 62, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 62, 62, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 31, 31, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 61504)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 70)             │     4,305,350 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           355 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,329,289 (16.51 MB)

 Trainable params: 4,329,289 (16.51 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
model.compile(optimizer="adam" , loss="mean_squared_error")

In [8]:
model.fit(train , validation_data=test , epochs=1)

2026-02-24 20:45:11.077311: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


313/313 ━━━━━━━━━━━━━━━━━━━━ 32s 98ms/step - loss: 6.7803 - val_loss: 5.7327


In [10]:
import cv2
import numpy as np

cap = cv2.VideoCapture(1)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # تبدیل BGR به RGB
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # resize برای مدل
    img = cv2.resize(img, (125,125))

    # normalize
    img = img.astype(np.float32) / 255.0

    # اضافه کردن batch
    img = np.expand_dims(img, axis=0)  # shape = (1,128,128,3)

    # پیش‌بینی
    pred = model.predict(img)[0]

    h, w = frame.shape[:2]

    ymin = int(pred[0] * h)
    xmin = int(pred[1] * w)
    ymax = int(pred[2] * h)
    xmax = int(pred[3] * w)

    # رسم باکس روی فریم اصلی
    cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), (0,255,0), 2)

    cv2.imshow("Detection", frame)

    key = cv2.waitKey(0) & 0xFF
    if key == ord(''):
        break

cap.release()
cv2.destroyAllWindows()

[ WARN:0@141.708] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video1): can't open camera by index


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step


QFontDatabase: Cannot find font directory /home/hooman/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/hooman/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/hooman/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/hooman/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/hooman/venv/lib/python3.12/site-packages/cv2

TypeError: ord() expected a character, but string of length 0 found